## Extrair os dados

In [1]:
# carregar o arquivo escolas10KLocDif.sql
from pathlib import Path

caminho_sql = Path("escolas10KLocDif.sql")

texto = caminho_sql.read_text(encoding="utf-8")

print(type(texto))
print(len(texto))
print(texto[:500])

<class 'str'>
2097798
DROP TABLE IF EXISTS telefone;
DROP TABLE IF EXISTS escola;

CREATE TABLE escola (
  codigo INT PRIMARY KEY,
  nome 	 TEXT NOT NULL,
  cod_municipio	 INT NOT NULL,
  endereco TEXT NULL,
  compl_endereco TEXT NULL,
  bairro TEXT NULL,
  cod_loc_dif INT NULL,
  FOREIGN KEY (cod_municipio) REFERENCES municipio(codigo),
  FOREIGN KEY (cod_loc_dif) REFERENCES localizacao_dif(codigo)
);


INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11022558, 


In [2]:
#transformar o texto em uma lista de linhas
linhas = texto.splitlines()
print(type(linhas))
print(len(linhas))
print(linhas[:5])

<class 'list'>
10001
['DROP TABLE IF EXISTS telefone;', 'DROP TABLE IF EXISTS escola;', '', 'CREATE TABLE escola (', '  codigo INT PRIMARY KEY,']


In [3]:
# Depois filtre apenas os INSERT da tabela escola
inserts = [linha for linha in linhas if linha.startswith("INSERT INTO escola")]
print(type(inserts))
print(len(inserts))
print(inserts[:5])

<class 'list'>
9985
["INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11022558, 'EIEEF HAP BITT TUPARI', 1100015,  'TERRA INDIGENA RIO BRANCO', 'ALDEIA COLORADO', 'RURAL',2);", "INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11024275, 'CEEJA LUIZ VAZ DE CAMOES', 1100015,  'AVENIDA RIO DE JANEIRO', 'ESCOLA', 'CIDADE ALTA',0);", "INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11024291, 'EMMEF 7 DE SETEMBRO', 1100015,  'LINHA 60 COM A 140', NULL, NULL ,0);", "INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11024372, 'EMEIEF ANA NERY', 1100015,  'ROLIM DE MOURA DO GUAPORE', NULL, NULL ,0);", "INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11024666, 'EMEIEF BOA ESPERANCA', 1100015,  'LINHA P 50 KM 22', 'ZONA RURA

In [4]:
# Extrair os valores dos inserts usando expressões regulares
import re   
valores = []
for insert in inserts:
    match = re.search(r"VALUES\s*\((.*)\);", insert)
    if match:
        valores.append(match.group(1)) 
print(type(valores))
print(len(valores))
print(valores[:5])

<class 'list'>
9985
["11022558, 'EIEEF HAP BITT TUPARI', 1100015,  'TERRA INDIGENA RIO BRANCO', 'ALDEIA COLORADO', 'RURAL',2", "11024275, 'CEEJA LUIZ VAZ DE CAMOES', 1100015,  'AVENIDA RIO DE JANEIRO', 'ESCOLA', 'CIDADE ALTA',0", "11024291, 'EMMEF 7 DE SETEMBRO', 1100015,  'LINHA 60 COM A 140', NULL, NULL ,0", "11024372, 'EMEIEF ANA NERY', 1100015,  'ROLIM DE MOURA DO GUAPORE', NULL, NULL ,0", "11024666, 'EMEIEF BOA ESPERANCA', 1100015,  'LINHA P 50 KM 22', 'ZONA RURAL', 'LINHA P.50',0"]


# 

In [5]:
# ler valores com modulo csv
import csv

linhas_campos = list(
    csv.reader(
        valores,
        delimiter=",",
        quotechar="'",
        skipinitialspace=True
    )
)
campos = linhas_campos[0]

print(campos)
print(len(campos))
print(len(linhas_campos))

['11022558', 'EIEEF HAP BITT TUPARI', '1100015', 'TERRA INDIGENA RIO BRANCO', 'ALDEIA COLORADO', 'RURAL', '2']
7
9985


In [6]:
# Criar uma função de limpeza de campo retirar os espaços em branco e NULLs
def limpar_campo(campo):
    campo = campo.strip()  # Remove espaços em branco
    if campo.upper() == "NULL":
        return None  # Converte 'NULL' para None
    return campo
# Aplicar a função de limpeza em cada campo
linhas_limpa = []
for linha in linhas_campos:
    linha_limpa = [limpar_campo(campo) for campo in linha]
    linhas_limpa.append(linha_limpa)
print(linhas_limpa[:5])


[['11022558', 'EIEEF HAP BITT TUPARI', '1100015', 'TERRA INDIGENA RIO BRANCO', 'ALDEIA COLORADO', 'RURAL', '2'], ['11024275', 'CEEJA LUIZ VAZ DE CAMOES', '1100015', 'AVENIDA RIO DE JANEIRO', 'ESCOLA', 'CIDADE ALTA', '0'], ['11024291', 'EMMEF 7 DE SETEMBRO', '1100015', 'LINHA 60 COM A 140', None, None, '0'], ['11024372', 'EMEIEF ANA NERY', '1100015', 'ROLIM DE MOURA DO GUAPORE', None, None, '0'], ['11024666', 'EMEIEF BOA ESPERANCA', '1100015', 'LINHA P 50 KM 22', 'ZONA RURAL', 'LINHA P.50', '0']]


In [7]:
#Verificar se todas as linhas têm 7 campos
tamanhos = []

for linha in linhas_limpa:
    tamanhos.append(len(linha))

set(tamanhos)

{7}

In [8]:
# criar o DataFrame com pandas com as colunas   codigo, nome, cod_municipio,
#    endereco, compl_endereco,bairro, cod_loc_dif
import pandas as pd
colunas = [
    "codigo",
    "nome",
    "cod_municipio",
    "endereco",
    "compl_endereco",
    "bairro",
    "cod_loc_dif"
]
df = pd.DataFrame(linhas_limpa, columns=colunas)
print(df.head())
print(df.info())

     codigo                      nome cod_municipio  \
0  11022558     EIEEF HAP BITT TUPARI       1100015   
1  11024275  CEEJA LUIZ VAZ DE CAMOES       1100015   
2  11024291       EMMEF 7 DE SETEMBRO       1100015   
3  11024372           EMEIEF ANA NERY       1100015   
4  11024666      EMEIEF BOA ESPERANCA       1100015   

                    endereco   compl_endereco       bairro cod_loc_dif  
0  TERRA INDIGENA RIO BRANCO  ALDEIA COLORADO        RURAL           2  
1     AVENIDA RIO DE JANEIRO           ESCOLA  CIDADE ALTA           0  
2         LINHA 60 COM A 140              NaN          NaN           0  
3  ROLIM DE MOURA DO GUAPORE              NaN          NaN           0  
4           LINHA P 50 KM 22       ZONA RURAL   LINHA P.50           0  
<class 'pandas.DataFrame'>
RangeIndex: 9985 entries, 0 to 9984
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   codigo          9985 non-null   str  
 1   no

In [9]:
# converer as colunas codigo, cod_municipio e cod_loc_dif para numero  
df["codigo"] = pd.to_numeric(df["codigo"], errors="coerce").astype("Int64")
df["cod_municipio"] = pd.to_numeric(df["cod_municipio"], errors="coerce").astype("Int64")
df["cod_loc_dif"] = pd.to_numeric(df["cod_loc_dif"], errors="coerce").astype("Int64")
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 9985 entries, 0 to 9984
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   codigo          9985 non-null   Int64
 1   nome            9985 non-null   str  
 2   cod_municipio   9985 non-null   Int64
 3   endereco        9982 non-null   str  
 4   compl_endereco  5485 non-null   str  
 5   bairro          6578 non-null   str  
 6   cod_loc_dif     8715 non-null   Int64
dtypes: Int64(3), str(4)
memory usage: 575.4 KB
None


In [10]:
#Limpar strings no DataFrame removendo espacos em branco e trocando strings vazias por NA
df["nome"] = df["nome"].str.strip().replace("", pd.NA)
df["endereco"] = df["endereco"].str.strip().replace("", pd.NA)
df["compl_endereco"] = df["compl_endereco"].str.strip().replace("", pd.NA)
df["bairro"] = df["bairro"].str.strip().replace("", pd.NA)
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 9985 entries, 0 to 9984
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   codigo          9985 non-null   Int64
 1   nome            9985 non-null   str  
 2   cod_municipio   9985 non-null   Int64
 3   endereco        9982 non-null   str  
 4   compl_endereco  5485 non-null   str  
 5   bairro          6578 non-null   str  
 6   cod_loc_dif     8715 non-null   Int64
dtypes: Int64(3), str(4)
memory usage: 575.4 KB
None


In [11]:
# Salvar em CSV e depois ler o CSV para verificar se os dados estão corretos
df.to_csv("escolas10KLocDif.csv", index=False)
df_csv = pd.read_csv("escolas10KLocDif.csv")
print(df_csv.info())
print(df_csv.head())    

<class 'pandas.DataFrame'>
RangeIndex: 9985 entries, 0 to 9984
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   codigo          9985 non-null   int64  
 1   nome            9985 non-null   str    
 2   cod_municipio   9985 non-null   int64  
 3   endereco        9982 non-null   str    
 4   compl_endereco  5485 non-null   str    
 5   bairro          6578 non-null   str    
 6   cod_loc_dif     8715 non-null   float64
dtypes: float64(1), int64(2), str(4)
memory usage: 546.2 KB
None
     codigo                      nome  cod_municipio  \
0  11022558     EIEEF HAP BITT TUPARI        1100015   
1  11024275  CEEJA LUIZ VAZ DE CAMOES        1100015   
2  11024291       EMMEF 7 DE SETEMBRO        1100015   
3  11024372           EMEIEF ANA NERY        1100015   
4  11024666      EMEIEF BOA ESPERANCA        1100015   

                    endereco   compl_endereco       bairro  cod_loc_dif  
0  TERRA INDIGENA RIO BR

In [12]:
#criar uma função para ler outros Sql e retornar um DataFrame dando um tratamento semelhante ao que foi feito acima
# A função deve verificar o nome no arquivo sql o nome da tavela e os campos e retornar um DataFrame com os dados tratados 
def ler_sql_para_dataframe(caminho_sql):
    from pathlib import Path
    import csv
    import re

    import pandas as pd

    caminho_sql = Path(caminho_sql)
    texto = caminho_sql.read_text(encoding="utf-8")

    create_table = re.search(
        r"CREATE\s+TABLE\s+(\w+)\s*\((.*?)\);",
        texto,
        flags=re.IGNORECASE | re.DOTALL
    )
    if create_table is None:
        raise ValueError("Nao foi encontrado CREATE TABLE no arquivo SQL.")

    nome_tabela = create_table.group(1)
    definicao_campos = create_table.group(2)

    colunas = []
    tipos = {}
    for linha in definicao_campos.splitlines():
        linha = linha.strip().rstrip(",")
        if not linha:
            continue

        primeira_palavra = linha.split()[0].upper()
        if primeira_palavra in {"FOREIGN", "PRIMARY", "UNIQUE", "CONSTRAINT", "CHECK"}:
            continue

        partes = linha.split()
        nome_coluna = partes[0].strip('`"[]')
        tipo_coluna = partes[1].upper() if len(partes) > 1 else "TEXT"
        colunas.append(nome_coluna)
        tipos[nome_coluna] = tipo_coluna

    padrao_insert = rf"INSERT\s+INTO\s+{nome_tabela}\s*\([^)]*\)\s*VALUES\s*\((.*)\);"
    valores = re.findall(padrao_insert, texto, flags=re.IGNORECASE)
    if not valores:
        raise ValueError(f"Nao foram encontrados INSERTs para a tabela {nome_tabela}.")

    linhas_campos = list(
        csv.reader(
            valores,
            delimiter=",",
            quotechar="'",
            skipinitialspace=True
        )
    )

    def limpar_campo(campo):
        campo = campo.strip()
        if campo.upper() == "NULL" or campo == "":
            return pd.NA
        return campo

    linhas_limpa = [
        [limpar_campo(campo) for campo in linha]
        for linha in linhas_campos
    ]

    df = pd.DataFrame(linhas_limpa, columns=colunas)

    for coluna, tipo in tipos.items():
        if "INT" in tipo:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce").astype("Int64")
        elif any(tipo_num in tipo for tipo_num in ["REAL", "FLOAT", "DOUBLE", "NUMERIC", "DECIMAL"]):
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
        else:
            df[coluna] = df[coluna].astype("string").str.strip().replace("", pd.NA)

    return df


df_municipios = ler_sql_para_dataframe("municipios.sql")
print(df_municipios.info())
print(df_municipios.head())


<class 'pandas.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   codigo  5570 non-null   Int64 
 1   nome    5570 non-null   string
 2   uf      5570 non-null   string
dtypes: Int64(1), string(2)
memory usage: 136.1 KB
None
    codigo                   nome  uf
0  1100015  Alta Floresta D'Oeste  RO
1  1100023              Ariquemes  RO
2  1100031                 Cabixi  RO
3  1100049                 Cacoal  RO
4  1100056             Cerejeiras  RO


#### Crie um código que exibe a quantidade de caracteres e o nome da escola com maior número de caracteres para cada um dos estados.

In [13]:
#carregar os arquivos de estados,municipios e regioes para fazer um merge e enriquecer os dados de escolas com o nome do municipio, estado e regiao
df_estados = ler_sql_para_dataframe("estados.sql")
#print(df_estados.info())
#print(df_estados.head())
df_regioes = ler_sql_para_dataframe("regioes.sql")
#print(df_regioes.info())
#print(df_regioes.head())
df_municipios = ler_sql_para_dataframe("municipios.sql")
#print(df_municipios.info())
#print(df_municipios.head())
df_escolas = ler_sql_para_dataframe("escolas10KLocDif.sql")
#print(df_escolas.info())
#print(df_escolas.head())
df_merge = df_escolas.merge(df_municipios, left_on="cod_municipio", right_on="codigo", how="left", suffixes=("", "_municipio")).drop(columns=["codigo_municipio"])  
df_merge = df_merge.merge(df_estados, left_on="uf", right_on="uf", how="left", suffixes=("", "_estado"))
df_merge = df_merge.merge(df_regioes, left_on="regiao", right_on="sigla", how="left", suffixes=("", "_regiao")).drop(columns=["sigla"])
print(df_merge.info())
print(df_merge.head())

<class 'pandas.DataFrame'>
RangeIndex: 9985 entries, 0 to 9984
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   codigo          9985 non-null   Int64 
 1   nome            9985 non-null   string
 2   cod_municipio   9985 non-null   Int64 
 3   endereco        9982 non-null   string
 4   compl_endereco  5485 non-null   string
 5   bairro          6578 non-null   string
 6   cod_loc_dif     8715 non-null   Int64 
 7   nome_municipio  9985 non-null   string
 8   uf              9985 non-null   string
 9   nome_estado     9985 non-null   string
 10  regiao          9985 non-null   string
 11  nome_regiao     9985 non-null   string
dtypes: Int64(3), string(9)
memory usage: 965.5 KB
None
     codigo                      nome  cod_municipio  \
0  11022558     EIEEF HAP BITT TUPARI        1100015   
1  11024275  CEEJA LUIZ VAZ DE CAMOES        1100015   
2  11024291       EMMEF 7 DE SETEMBRO        1100015   
3  11024

In [14]:
#criar coluna com numero de caracteres do nome da escola
df_merge["tamanho_nome"] = df_merge["nome"].str.len()
print(df_merge[["nome", "tamanho_nome"]].head())

                       nome  tamanho_nome
0     EIEEF HAP BITT TUPARI            21
1  CEEJA LUIZ VAZ DE CAMOES            24
2       EMMEF 7 DE SETEMBRO            19
3           EMEIEF ANA NERY            15
4      EMEIEF BOA ESPERANCA            20


In [15]:
# mostrar a escola com o nome mais longo
escola_nome_mais_longo = df_merge.loc[df_merge["tamanho_nome"].idxmax()]
print(escola_nome_mais_longo[["nome", "tamanho_nome"]])

nome            CENTRO MUNICIPAL DE EDUCACAO INFANTIL DOS SABE...
tamanho_nome                                                  100
Name: 3624, dtype: object


#### Exibir o número de escolas em cada cidade. Ordene a consulta pelo nome do estado, seguido pelo número de escolas, de forma decrescente.

In [16]:
# Criar uma tabela agrupada por nome de cidade e nome do estado consolidado
# numero de escolas por cidade
df_agrupado = df_merge.groupby(["nome_municipio", "nome_estado"]).size().reset_index(name="numero_escolas")
print(df_agrupado.head())

            nome_municipio nome_estado  numero_escolas
0               Acrelândia        Acre              12
1    Alta Floresta D'Oeste    Rondônia              36
2              Alto Alegre     Roraima              52
3  Alto Alegre dos Parecis    Rondônia              14
4             Alto Paraíso    Rondônia              15


In [17]:
# ordenar de forma decrescente pelo nome do estado e numero de escolas por cidade
df_agrupado_ordenado = df_agrupado.sort_values(by=["nome_estado", "numero_escolas"], ascending=[False, False])
print(df_agrupado_ordenado)

        nome_municipio nome_estado  numero_escolas
20           Boa Vista     Roraima             273
7              Amajari     Roraima              54
34               Cantá     Roraima              53
2          Alto Alegre     Roraima              52
23              Bonfim     Roraima              41
..                 ...         ...             ...
105         Porto Acre        Acre              32
104  Plácido de Castro        Acre              25
54      Epitaciolândia        Acre              23
36            Capixaba        Acre              15
0           Acrelândia        Acre              12

[145 rows x 3 columns]


#### Exibir a quantidade de escolas por região, e  ordene os resultados por número de escolas.

In [19]:
# Quantidade total de escolas por regiao, ordenada pelo numero de escolas
df_regiao_agrupada = (
    df_merge.groupby("nome_regiao")
    .size()
    .reset_index(name="numero_escolas")
    .sort_values(by="numero_escolas", ascending=False)
)

print(df_regiao_agrupada)


  nome_regiao  numero_escolas
0       Norte            9985


#### exibir a quantidade de escolas sem o dado de endereço, por região, e ordene os resultados por número de escolas.

In [20]:
# Quantidade de escolas sem dado de endereco por regiao, ordenada pelo numero de escolas
df_sem_endereco_por_regiao = (
    df_merge[df_merge["endereco"].isna()]
    .groupby("nome_regiao")
    .size()
    .reset_index(name="numero_escolas_sem_endereco")
    .sort_values(by="numero_escolas_sem_endereco", ascending=False)
)

print(df_sem_endereco_por_regiao)


  nome_regiao  numero_escolas_sem_endereco
0       Norte                            3
